# Sentiment Analysis — BERT Model with 4-Fold Domain Cross Validation

**Dataset:** Multi-Domain Amazon Reviews (Books, DVD, Electronics, Kitchen)

**Strategy:** Each domain takes a turn as the test set while the other 3 are used for reference.

| Round | Train (Reference) | Test |
|---|---|---|
| Round 1 | DVD + Electronics + Kitchen | **Books** |
| Round 2 | Books + Electronics + Kitchen | **DVD** |
| Round 3 | Books + DVD + Kitchen | **Electronics** |
| Round 4 | Books + DVD + Electronics | **Kitchen** |


## Step 1 — Install Dependencies

In [1]:
!pip install -q transformers torch nltk scikit-learn


## Step 2 — Import Libraries

In [2]:
import os, re, nltk, urllib.request, tarfile
import numpy as np
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import pipeline

nltk.download('stopwords')
nltk.download('punkt')

device = 0 if torch.cuda.is_available() else -1
print('Libraries imported successfully!')
print(f'Device: {"GPU" if device == 0 else "CPU"}')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Libraries imported successfully!
Device: GPU


## Step 3 — Download Dataset

In [3]:
url          = 'https://www.cs.jhu.edu/~mdredze/datasets/sentiment/domain_sentiment_data.tar.gz'
tar_path     = 'domain_sentiment_data.tar.gz'
extract_path = 'sentiment_data'

if not os.path.exists(extract_path):
    print('Downloading dataset... (~30MB)')
    urllib.request.urlretrieve(url, tar_path)
    print('Extracting...')
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(extract_path, filter='data')
    print('Done!')
else:
    print('Dataset already downloaded.')

path = f'{extract_path}/sorted_data_acl/'
print('Folders:', os.listdir(path))


Extracting...
Done!
Folders: ['books', 'electronics', 'kitchen_&_housewares', 'dvd']


## Step 4 — Text Cleaning

In [4]:
def clean_sentence(sentence: str) -> str:
    sentence = re.sub(r'(<review_text>|<\/review_text>)', '', sentence)
    sentence = sentence.lower()
    sentence = re.sub(r'\bhttp\S+|\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', '', sentence)
    sentence = re.sub(r'@', 'a', sentence)
    sentence = re.sub(r'[^\w\s\-]', '', sentence)
    sentence = re.sub(r'\s+', ' ', sentence).strip()
    return sentence

print('Before:', '<review_text>This product is GREAT!!! Visit www.amazon.com</review_text>')
print('After :', clean_sentence('<review_text>This product is GREAT!!! Visit www.amazon.com</review_text>'))


Before: <review_text>This product is GREAT!!! Visit www.amazon.com</review_text>
After : this product is great visit wwwamazoncom


## Step 5 — Load Dataset

In [5]:
regex_review = re.compile(r'.+?<\/review_text>', flags=re.DOTALL)

def read_reviews(folders, path, samples_per_class=500):
    x, y = [], []
    for folder in folders:
        for label, fname in [(0, 'negative.review'), (1, 'positive.review')]:
            raw     = open(path + folder + '/' + fname, 'r', encoding='utf-8', errors='ignore').read()
            reviews = re.findall(regex_review, raw)[:samples_per_class]
            sentiment = 'Negative' if label == 0 else 'Positive'
            print(f'  {len(reviews)} {sentiment} from [{folder}]')
            for s in reviews:
                x.append(clean_sentence(s))
                y.append(label)
    return x, y

DOMAINS = ['books', 'dvd', 'electronics', 'kitchen_&_housewares']

print('Loading all domain data...')
all_data = {}
for domain in DOMAINS:
    print(f'\n--- {domain.upper()} ---')
    x, y = read_reviews([domain], path, samples_per_class=500)
    all_data[domain] = (x, y)
    print(f'  Total: {len(x)} reviews')

print('\nAll domains loaded successfully!')


Loading all domain data...

--- BOOKS ---
  500 Negative from [books]
  500 Positive from [books]
  Total: 1000 reviews

--- DVD ---
  500 Negative from [dvd]
  500 Positive from [dvd]
  Total: 1000 reviews

--- ELECTRONICS ---
  500 Negative from [electronics]
  500 Positive from [electronics]
  Total: 1000 reviews

--- KITCHEN_&_HOUSEWARES ---
  500 Negative from [kitchen_&_housewares]
  500 Positive from [kitchen_&_housewares]
  Total: 1000 reviews

All domains loaded successfully!


## Step 6 — Load BERT Model

Using `textattack/bert-base-uncased-SST-2`.

> ⚠️ **Important fix:** This model returns `LABEL_0` (Negative) and `LABEL_1` (Positive)
> instead of the words NEGATIVE/POSITIVE. The predict function below handles this correctly.

> Enable GPU: **Runtime → Change runtime type → T4 GPU**


In [6]:
print('Loading BERT model (~440MB first download)...')

bert_model = pipeline(
    'sentiment-analysis',
    model='textattack/bert-base-uncased-SST-2',
    truncation=True,
    max_length=512,
    device=device
)

# Check what labels this model actually returns
test_output = bert_model('I love this product')
print('Model loaded!')
print('Label format this model uses:', test_output)
print('(LABEL_1 = Positive, LABEL_0 = Negative)')


Loading BERT model (~440MB first download)...


config.json:   0%|          | 0.00/477 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

model.safetensors: downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Model loaded!
Label format this model uses: [{'label': 'LABEL_1', 'score': 0.9995694756507874}]
(LABEL_1 = Positive, LABEL_0 = Negative)


## Step 7 — Predict Function (Fixed)

This correctly handles `LABEL_0` / `LABEL_1` output from this BERT model.


In [7]:
def bert_predict(review_text: str) -> int:
    """
    Returns 1 for POSITIVE, 0 for NEGATIVE.
    Handles both label formats:
      - 'LABEL_1' / 'LABEL_0' (textattack model)
      - 'POSITIVE' / 'NEGATIVE' (other models)
    """
    if not review_text.strip():
        return 0
    result = bert_model(review_text)[0]
    label  = result['label'].upper()
    # Handle LABEL_0 / LABEL_1 format
    if label == 'LABEL_1':
        return 1
    elif label == 'LABEL_0':
        return 0
    # Handle POSITIVE / NEGATIVE format
    elif 'POS' in label:
        return 1
    else:
        return 0

# Verify the fix works correctly
print('Testing predict function:')
print('Positive test ->', bert_predict('I absolutely love this, highly recommended!'))
print('Negative test ->', bert_predict('Terrible product, complete waste of money'))
print('Expected: 1 then 0')


Testing predict function:
Positive test -> 1
Negative test -> 0
Expected: 1 then 0


## Step 8 — Results Printer

In [8]:
def print_results(round_num, test_domain, train_domains, y_true, y_pred):
    acc    = accuracy_score(y_true, y_pred)
    cm     = confusion_matrix(y_true, y_pred)
    report = classification_report(
        y_true, y_pred,
        labels=[0, 1],
        target_names=['Negative', 'Positive'],
        zero_division=0
    )
    print('\n' + '=' * 60)
    print(f'  ROUND {round_num} RESULTS')
    print('=' * 60)
    print(f'  Train Domains : {", ".join(train_domains)}')
    print(f'  Test Domain   : {test_domain}')
    print(f'  BERT Accuracy : {acc * 100:.2f}%')
    print('=' * 60)
    print('\nClassification Report:')
    print(report)
    print('Confusion Matrix:')
    print(cm)
    print(f'  True Negatives  : {cm[0][0]}')
    print(f'  False Positives : {cm[0][1]}')
    print(f'  False Negatives : {cm[1][0]}')
    print(f'  True Positives  : {cm[1][1]}')
    print('=' * 60)
    return acc


## Step 9 — Round 1: Test on BOOKS

**Train:** DVD + Electronics + Kitchen → **Test:** Books


In [9]:
TEST_DOMAIN_r1   = 'books'
TRAIN_DOMAINS_r1 = [d for d in DOMAINS if d != TEST_DOMAIN_r1]

x_test_r1, y_test_r1 = all_data[TEST_DOMAIN_r1]

print(f'Round 1: Testing on [{TEST_DOMAIN_r1.upper()}]')
print(f'Train domains: {TRAIN_DOMAINS_r1}')
print(f'Test samples : {len(x_test_r1)}')
print('\nRunning BERT predictions...')

y_pred_r1 = []
for i_r, review in enumerate(x_test_r1):
    pred = bert_predict(review)
    y_pred_r1.append(pred)
    if (i_r + 1) % 100 == 0 or (i_r + 1) == len(x_test_r1):
        print(f'  [{i_r+1}/{len(x_test_r1)}] processed...')

acc_r1 = print_results(1, TEST_DOMAIN_r1, TRAIN_DOMAINS_r1, y_test_r1, y_pred_r1)


Round 1: Testing on [BOOKS]
Train domains: ['dvd', 'electronics', 'kitchen_&_housewares']
Test samples : 1000

Running BERT predictions...


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  [100/1000] processed...
  [200/1000] processed...
  [300/1000] processed...
  [400/1000] processed...
  [500/1000] processed...
  [600/1000] processed...
  [700/1000] processed...
  [800/1000] processed...
  [900/1000] processed...
  [1000/1000] processed...

  ROUND 1 RESULTS
  Train Domains : dvd, electronics, kitchen_&_housewares
  Test Domain   : books
  BERT Accuracy : 88.00%

Classification Report:
              precision    recall  f1-score   support

    Negative       0.87      0.89      0.88       500
    Positive       0.89      0.87      0.88       500

    accuracy                           0.88      1000
   macro avg       0.88      0.88      0.88      1000
weighted avg       0.88      0.88      0.88      1000

Confusion Matrix:
[[444  56]
 [ 64 436]]
  True Negatives  : 444
  False Positives : 56
  False Negatives : 64
  True Positives  : 436


## Step 10 — Round 2: Test on DVD

**Train:** Books + Electronics + Kitchen → **Test:** Dvd


In [10]:
TEST_DOMAIN_r2   = 'dvd'
TRAIN_DOMAINS_r2 = [d for d in DOMAINS if d != TEST_DOMAIN_r2]

x_test_r2, y_test_r2 = all_data[TEST_DOMAIN_r2]

print(f'Round 2: Testing on [{TEST_DOMAIN_r2.upper()}]')
print(f'Train domains: {TRAIN_DOMAINS_r2}')
print(f'Test samples : {len(x_test_r2)}')
print('\nRunning BERT predictions...')

y_pred_r2 = []
for i_r, review in enumerate(x_test_r2):
    pred = bert_predict(review)
    y_pred_r2.append(pred)
    if (i_r + 1) % 100 == 0 or (i_r + 1) == len(x_test_r2):
        print(f'  [{i_r+1}/{len(x_test_r2)}] processed...')

acc_r2 = print_results(2, TEST_DOMAIN_r2, TRAIN_DOMAINS_r2, y_test_r2, y_pred_r2)


Round 2: Testing on [DVD]
Train domains: ['books', 'electronics', 'kitchen_&_housewares']
Test samples : 1000

Running BERT predictions...
  [100/1000] processed...
  [200/1000] processed...
  [300/1000] processed...
  [400/1000] processed...
  [500/1000] processed...
  [600/1000] processed...
  [700/1000] processed...
  [800/1000] processed...
  [900/1000] processed...
  [1000/1000] processed...

  ROUND 2 RESULTS
  Train Domains : books, electronics, kitchen_&_housewares
  Test Domain   : dvd
  BERT Accuracy : 88.60%

Classification Report:
              precision    recall  f1-score   support

    Negative       0.85      0.94      0.89       500
    Positive       0.94      0.83      0.88       500

    accuracy                           0.89      1000
   macro avg       0.89      0.89      0.89      1000
weighted avg       0.89      0.89      0.89      1000

Confusion Matrix:
[[472  28]
 [ 86 414]]
  True Negatives  : 472
  False Positives : 28
  False Negatives : 86
  True Positi

## Step 11 — Round 3: Test on ELECTRONICS

**Train:** Books + DVD + Kitchen → **Test:** Electronics


In [11]:
TEST_DOMAIN_r3   = 'electronics'
TRAIN_DOMAINS_r3 = [d for d in DOMAINS if d != TEST_DOMAIN_r3]

x_test_r3, y_test_r3 = all_data[TEST_DOMAIN_r3]

print(f'Round 3: Testing on [{TEST_DOMAIN_r3.upper()}]')
print(f'Train domains: {TRAIN_DOMAINS_r3}')
print(f'Test samples : {len(x_test_r3)}')
print('\nRunning BERT predictions...')

y_pred_r3 = []
for i_r, review in enumerate(x_test_r3):
    pred = bert_predict(review)
    y_pred_r3.append(pred)
    if (i_r + 1) % 100 == 0 or (i_r + 1) == len(x_test_r3):
        print(f'  [{i_r+1}/{len(x_test_r3)}] processed...')

acc_r3 = print_results(3, TEST_DOMAIN_r3, TRAIN_DOMAINS_r3, y_test_r3, y_pred_r3)


Round 3: Testing on [ELECTRONICS]
Train domains: ['books', 'dvd', 'kitchen_&_housewares']
Test samples : 1000

Running BERT predictions...
  [100/1000] processed...
  [200/1000] processed...
  [300/1000] processed...
  [400/1000] processed...
  [500/1000] processed...
  [600/1000] processed...
  [700/1000] processed...
  [800/1000] processed...
  [900/1000] processed...
  [1000/1000] processed...

  ROUND 3 RESULTS
  Train Domains : books, dvd, kitchen_&_housewares
  Test Domain   : electronics
  BERT Accuracy : 78.00%

Classification Report:
              precision    recall  f1-score   support

    Negative       0.70      0.99      0.82       500
    Positive       0.99      0.57      0.72       500

    accuracy                           0.78      1000
   macro avg       0.84      0.78      0.77      1000
weighted avg       0.84      0.78      0.77      1000

Confusion Matrix:
[[497   3]
 [217 283]]
  True Negatives  : 497
  False Positives : 3
  False Negatives : 217
  True Positi

## Step 12 — Round 4: Test on KITCHEN & HOUSEWARES

**Train:** Books + DVD + Electronics → **Test:** Kitchen & Housewares


In [12]:
TEST_DOMAIN_r4   = 'kitchen_&_housewares'
TRAIN_DOMAINS_r4 = [d for d in DOMAINS if d != TEST_DOMAIN_r4]

x_test_r4, y_test_r4 = all_data[TEST_DOMAIN_r4]

print(f'Round 4: Testing on [{TEST_DOMAIN_r4.upper()}]')
print(f'Train domains: {TRAIN_DOMAINS_r4}')
print(f'Test samples : {len(x_test_r4)}')
print('\nRunning BERT predictions...')

y_pred_r4 = []
for i_r, review in enumerate(x_test_r4):
    pred = bert_predict(review)
    y_pred_r4.append(pred)
    if (i_r + 1) % 100 == 0 or (i_r + 1) == len(x_test_r4):
        print(f'  [{i_r+1}/{len(x_test_r4)}] processed...')

acc_r4 = print_results(4, TEST_DOMAIN_r4, TRAIN_DOMAINS_r4, y_test_r4, y_pred_r4)


Round 4: Testing on [KITCHEN_&_HOUSEWARES]
Train domains: ['books', 'dvd', 'electronics']
Test samples : 1000

Running BERT predictions...
  [100/1000] processed...
  [200/1000] processed...
  [300/1000] processed...
  [400/1000] processed...
  [500/1000] processed...
  [600/1000] processed...
  [700/1000] processed...
  [800/1000] processed...
  [900/1000] processed...
  [1000/1000] processed...

  ROUND 4 RESULTS
  Train Domains : books, dvd, electronics
  Test Domain   : kitchen_&_housewares
  BERT Accuracy : 82.00%

Classification Report:
              precision    recall  f1-score   support

    Negative       0.74      0.98      0.85       500
    Positive       0.97      0.66      0.79       500

    accuracy                           0.82      1000
   macro avg       0.86      0.82      0.82      1000
weighted avg       0.86      0.82      0.82      1000

Confusion Matrix:
[[491   9]
 [171 329]]
  True Negatives  : 491
  False Positives : 9
  False Negatives : 171
  True Positi

## Step 13 — Final Summary Across All 4 Rounds

In [13]:
print('\n' + '=' * 60)
print('  FINAL SUMMARY — ALL 4 ROUNDS')
print('=' * 60)
results = [
    ('Round 1', 'Books',       acc_r1),
    ('Round 2', 'DVD',         acc_r2),
    ('Round 3', 'Electronics', acc_r3),
    ('Round 4', 'Kitchen',     acc_r4),
]
for rname, domain, acc in results:
    bar = '█' * int(acc * 30)
    print(f'  {rname} | Test: {domain:<12} | {acc*100:.2f}% {bar}')

avg = np.mean([acc_r1, acc_r2, acc_r3, acc_r4])
print('=' * 60)
print(f'  Average Accuracy : {avg * 100:.2f}%')
print('  Best Domain      :', results[np.argmax([r[2] for r in results])][1])
print('  Worst Domain     :', results[np.argmin([r[2] for r in results])][1])
print('=' * 60)



  FINAL SUMMARY — ALL 4 ROUNDS
  Round 1 | Test: Books        | 88.00% ██████████████████████████
  Round 2 | Test: DVD          | 88.60% ██████████████████████████
  Round 3 | Test: Electronics  | 78.00% ███████████████████████
  Round 4 | Test: Kitchen      | 82.00% ████████████████████████
  Average Accuracy : 84.15%
  Best Domain      : DVD
  Worst Domain     : Electronics


## Step 14 — Live Demo

In [14]:
def predict(review: str):
    cleaned = clean_sentence(review)
    result  = bert_predict(cleaned)
    label   = '✅ Positive Review' if result == 1 else '❌ Negative Review'
    print(f'Review : {review[:70]}')
    print(f'Result : {label}')
    print()

print('--- Live Demo ---\n')
predict('I really recommend this book, it changed my life')
predict('The electronics stopped working after 2 days, total waste of money')
predict('Amazing kitchen product, makes cooking so much easier')
predict('Worst dvd quality I have ever seen, very disappointed')

# Try your own:
# predict('Write your own review here')


--- Live Demo ---

Review : I really recommend this book, it changed my life
Result : ✅ Positive Review

Review : The electronics stopped working after 2 days, total waste of money
Result : ❌ Negative Review

Review : Amazing kitchen product, makes cooking so much easier
Result : ✅ Positive Review

Review : Worst dvd quality I have ever seen, very disappointed
Result : ❌ Negative Review

